# ⭐ Day 61: Model Evaluation & Cross-Validation Mastery
The Foundation of Reliable Machine Learning

*Day 61 of 369-day Python & AI Learning Path*

## Welcome to Day 61!

Welcome to one of the most critical days in your machine learning journey! 🎯 If you've ever trained a model that performed beautifully on your training data but fell apart in production, you already understand why proper evaluation matters. Today, we bridge the gap between "it works on my machine" and "it works reliably everywhere."

Model evaluation is the unsung hero of machine learning. While flashy algorithms and complex architectures get the headlines, it's the rigorous evaluation methodology that separates hobbyist projects from production-grade systems. A model is only as good as your ability to measure its true performance.

Throughout this notebook, we'll explore the full spectrum of evaluation techniques — from basic train-test splits to sophisticated cross-validation strategies. You'll learn when to use each metric, how to detect overfitting before it becomes a problem, and how to build evaluation pipelines that give you confidence in your models.

By the end of today, you'll have a comprehensive toolkit for reliably estimating model performance, comparing different approaches, and making data-driven decisions about which model to deploy. Let's dive in! 🚀

## 📋 Table of Contents

1. [Why Proper Model Evaluation Matters](#1-why-proper-model-evaluation-matters)
2. [Train-Test Split vs Cross-Validation](#2-train-test-split-vs-cross-validation)
3. [Cross-Validation Strategies](#3-cross-validation-strategies)
4. [Evaluation Metrics Deep Dive](#4-evaluation-metrics-deep-dive)
5. [Implementing Cross-Validation with Scikit-Learn](#5-implementing-cross-validation-with-scikit-learn)
6. [Applying to Titanic Dataset (Classification)](#6-applying-to-titanic-dataset-classification)
7. [Applying to House Prices Dataset (Regression)](#7-applying-to-house-prices-dataset-regression)
8. [Overfitting vs Underfitting Detection](#8-overfitting-vs-underfitting-detection)
9. [Best Practices for Reliable Evaluation](#9-best-practices-for-reliable-evaluation)
10. [Nested Cross-Validation](#10-nested-cross-validation)
11. [🛠️ Hands-On Exercises](#hands-on-exercises)
12. [Solutions](#solutions)

---

## 1. Why Proper Model Evaluation Matters 💡

Proper model evaluation is the cornerstone of trustworthy machine learning. Without it, we risk:

- **Overfitting to training data**: Models that memorize rather than generalize
- **Optimistic performance estimates**: Believing a model is better than it actually is
- **Poor production performance**: Deploying models that fail on real-world data
- **Wasted resources**: Investing in models that don't solve the actual problem

### Key Principles:
1. **Generalization is the goal** — we care about performance on unseen data
2. **Data leakage is the enemy** — never let test information influence training
3. **Multiple metrics tell the full story** — no single metric captures everything
4. **Statistical rigor matters** — confidence intervals and variance are as important as point estimates

In [ ]:
# 📦 Standard imports for the entire notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    KFold, StratifiedKFold, LeaveOneOut, TimeSeriesSplit,
    learning_curve, validation_curve, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✅ All libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {__import__('sklearn').__version__}")

---

## 2. Train-Test Split vs Cross-Validation 🔄

### Train-Test Split
The simplest evaluation approach: divide data into training and testing sets.

**Pros:**
- Fast and simple
- Easy to understand and implement
- Good for large datasets

**Cons:**
- Performance estimate depends heavily on how the split is made
- High variance in performance estimates
- Wastes data that could be used for training

### Cross-Validation
Systematically rotate training and testing sets across multiple splits.

**Pros:**
- More reliable performance estimates
- Better use of available data
- Provides variance estimates
- Less sensitive to data partitioning

**Cons:**
- Computationally more expensive
- More complex to implement and interpret

In [ ]:
# 📊 Demonstrating variance in train-test split vs stability of cross-validation
from sklearn.datasets import make_classification

# Create a synthetic dataset
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           n_redundant=3, random_state=42)

model = LogisticRegression(max_iter=1000, random_state=42)

# Multiple random train-test splits
split_scores = []
for seed in range(20):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    model.fit(X_train, y_train)
    split_scores.append(model.score(X_test, y_test))

# Cross-validation scores
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Train-test split variance
axes[0].plot(range(1, 21), split_scores, 'o-', color='coral', linewidth=2, markersize=6)
axes[0].axhline(np.mean(split_scores), color='darkred', linestyle='--', label=f'Mean: {np.mean(split_scores):.3f}')
axes[0].fill_between(range(1, 21), np.mean(split_scores) - np.std(split_scores),
                     np.mean(split_scores) + np.std(split_scores), alpha=0.2, color='coral')
axes[0].set_title('📏 Train-Test Split: High Variance', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Random Seed')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].set_ylim(0.5, 1.0)
axes[0].grid(True, alpha=0.3)

# Plot 2: Cross-validation stability
axes[1].bar(range(1, 6), cv_scores, color='steelblue', edgecolor='navy', alpha=0.8)
axes[1].axhline(np.mean(cv_scores), color='darkgreen', linestyle='--', linewidth=2,
                label=f'Mean: {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f}')
axes[1].set_title('🔄 5-Fold CV: Stable & Reliable', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Fold Number')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_ylim(0.5, 1.0)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Train-Test Split vs Cross-Validation', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Train-Test Split — Mean: {np.mean(split_scores):.4f}, Std: {np.std(split_scores):.4f}")
print(f"5-Fold CV          — Mean: {np.mean(cv_scores):.4f}, Std: {np.std(cv_scores):.4f}")
print(f"\n💡 Cross-validation provides a more stable estimate with lower variance!")

---

## 3. Cross-Validation Strategies 🔄

Different problems require different cross-validation strategies. Choosing the right one is crucial for reliable evaluation.

### 3.1 K-Fold Cross-Validation
The classic approach: split data into K equal folds, train on K-1, test on 1.

### 3.2 Stratified K-Fold
Preserves class distribution in each fold — essential for imbalanced datasets.

### 3.3 Leave-One-Out (LOO)
Extreme case where K = N (number of samples). Very unbiased but high variance.

### 3.4 Time Series Split
Respects temporal ordering — critical for time-dependent data.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut, TimeSeriesSplit
from matplotlib.patches import Patch

# --- FIX APPLIED HERE ---
# Added n_informative=2 and n_redundant=0 to fit within n_features=2
X_small, y_small = make_classification(
    n_samples=12, 
    n_features=2, 
    n_informative=2, 
    n_redundant=0, 
    n_classes=2,
    weights=[0.67, 0.33], 
    random_state=42
)

strategies = {
    'K-Fold (K=3)': KFold(n_splits=3, shuffle=True, random_state=42),
    'Stratified K-Fold (K=3)': StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    'Leave-One-Out': LeaveOneOut(),
    'Time Series Split (K=3)': TimeSeriesSplit(n_splits=3)
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, cv) in enumerate(strategies.items()):
    ax = axes[idx]
    fold_idx = 0
    
    # Generate the splits
    for train_idx, test_idx in cv.split(X_small, y_small):
        y_pos = fold_idx
        for i in range(len(X_small)):
            if i in test_idx:
                ax.barh(y_pos, 1, left=i, color='steelblue', edgecolor='white', height=0.8)
            else:
                ax.barh(y_pos, 1, left=i, color='lightcoral', edgecolor='white', height=0.8, alpha=0.7)
        fold_idx += 1

    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Fold')
    ax.set_yticks(range(fold_idx))
    ax.set_yticklabels([f'Fold {i+1}' for i in range(fold_idx)])
    ax.set_xlim(0, len(X_small))
    ax.invert_yaxis()

    # Add legend
    legend_elements = [
        Patch(facecolor='lightcoral', label='Train'),
        Patch(facecolor='steelblue', label='Test')
    ]
    ax.legend(handles=legend_elements, loc='upper right')

plt.suptitle('Cross-Validation Strategies Visualized', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📊 Blue = Test set, Red = Training set for each fold")
print("\n💡 Key Takeaways:")
print("   • K-Fold: Simple, works well for most cases")
print("   • Stratified K-Fold: Preserves class balance — use for imbalanced data")
print("   • Leave-One-Out: Best for very small datasets, computationally expensive")
print("   • Time Series Split: Never shuffles — respects temporal order")

In [ ]:
# 📏 Comparing CV strategies on an imbalanced dataset
from sklearn.datasets import make_classification

# Create imbalanced dataset
X_imb, y_imb = make_classification(n_samples=1000, n_features=20, n_classes=2,
                                    weights=[0.9, 0.1], random_state=42)

model = LogisticRegression(max_iter=1000, random_state=42)

# Compare strategies
strategies_comparison = {
    'K-Fold (5)': KFold(n_splits=5, shuffle=True, random_state=42),
    'Stratified K-Fold (5)': StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'K-Fold (10)': KFold(n_splits=10, shuffle=True, random_state=42),
    'Stratified K-Fold (10)': StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
}

results = {}
for name, cv in strategies_comparison.items():
    scores = cross_val_score(model, X_imb, y_imb, cv=cv, scoring='f1')
    results[name] = scores

# Boxplot comparison
fig, ax = plt.subplots(figsize=(12, 6))
bp = ax.boxplot(results.values(), labels=results.keys(), patch_artist=True,
                notch=True, showmeans=True, meanline=True)

colors = ['lightcoral', 'lightgreen', 'coral', 'green']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('📊 F1-Score Comparison: K-Fold vs Stratified K-Fold\n(Imbalanced Dataset)', 
             fontsize=14, fontweight='bold')
ax.set_ylabel('F1-Score')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Print statistics
print("📏 Strategy Comparison (F1-Score):")
print("-" * 50)
for name, scores in results.items():
    print(f"{name:25s}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
print("\n💡 Stratified K-Fold maintains more consistent performance on imbalanced data!")

---

## 4. Evaluation Metrics Deep Dive 📊

### 4.1 Regression Metrics

| Metric | Formula | Best For | Sensitive To |
|--------|---------|----------|--------------|
| **MAE** | mean(\|y - ŷ\|) | Interpretability | Outliers (linear) |
| **MSE** | mean((y - ŷ)²) | Optimization | Outliers (quadratic) |
| **RMSE** | √MSE | Same units as target | Outliers (quadratic) |
| **R²** | 1 - SS_res/SS_tot | Variance explained | Scale-independent |

### 4.2 Classification Metrics

| Metric | Formula | Best For | Watch Out For |
|--------|---------|----------|---------------|
| **Accuracy** | (TP+TN)/(Total) | Balanced classes | Imbalanced data |
| **Precision** | TP/(TP+FP) | Minimize false positives | Cost of FP |
| **Recall** | TP/(TP+FN) | Minimize false negatives | Cost of FN |
| **F1-Score** | 2·P·R/(P+R) | Balance precision & recall | Class imbalance |
| **ROC-AUC** | Area under ROC | Threshold-independent | Severe imbalance |

### Confusion Matrix
The foundation of all classification metrics:
```
                 Predicted
              0      1
Actual   0   TN     FP
         1   FN     TP
```

In [ ]:
# 📊 Comprehensive Classification Metrics Demonstration
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Create a classification dataset with some imbalance
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2,
                           weights=[0.7, 0.3], random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate all metrics
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
}

print("📊 Classification Metrics Report")
print("=" * 40)
for metric, value in metrics.items():
    print(f"{metric:12s}: {value:.4f}")
print("\n" + classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))

In [ ]:
# 📊 Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title('📊 Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')

# Normalized heatmap
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Oranges', ax=axes[1],
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[1].set_title('📊 Confusion Matrix (Normalized by Row)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Actual Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

# Print interpretation
tn, fp, fn, tp = cm.ravel()
print(f"\n💡 Confusion Matrix Interpretation:")
print(f"   True Negatives (TN):  {tn:4d} — Correctly predicted Class 0")
print(f"   False Positives (FP): {fp:4d} — Incorrectly predicted Class 1 (Type I error)")
print(f"   False Negatives (FN): {fn:4d} — Incorrectly predicted Class 0 (Type II error)")
print(f"   True Positives (TP):  {tp:4d} — Correctly predicted Class 1")
print(f"\n   Specificity (TNR): {tn/(tn+fp):.4f}")
print(f"   Sensitivity (TPR): {tp/(tp+fn):.4f}")

In [ ]:
# 📈 ROC Curve and Precision-Recall Curve
fpr, tpr, roc_thresholds = roc_curve(y_test, y_pred_proba)
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {metrics["ROC-AUC"]:.3f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[0].fill_between(fpr, tpr, alpha=0.2, color='darkorange')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0].set_title('📈 ROC Curve', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
axes[1].plot(recall, precision, color='green', lw=2, label='PR Curve')
axes[1].fill_between(recall, precision, alpha=0.2, color='green')
baseline = np.sum(y_test) / len(y_test)
axes[1].axhline(baseline, color='red', linestyle='--', label=f'Baseline ({baseline:.3f})')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision')
axes[1].set_title('📈 Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Classification Performance Curves', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Key Insights:")
print("   • ROC-AUC = 0.5 → Random guessing")
print("   • ROC-AUC = 1.0 → Perfect classifier")
print("   • PR Curve is more informative than ROC for imbalanced data")
print("   • The closer the curve hugs the top-left, the better the model")

---

## 5. Implementing Cross-Validation with Scikit-Learn 🛠️

Scikit-Learn provides powerful tools for cross-validation:

- `cross_val_score`: Simple scoring across folds
- `cross_validate`: Multiple metrics and train scores
- `KFold`, `StratifiedKFold`, `LeaveOneOut`, `TimeSeriesSplit`: CV splitters

In [ ]:
# 🔄 cross_val_score vs cross_validate demonstration
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Simple cross_val_score
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print("📊 cross_val_score (Accuracy):")
print(f"   Scores: {cv_scores}")
print(f"   Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print()

# Advanced cross_validate with multiple metrics
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_results = cross_validate(model, X, y, cv=5, scoring=scoring, return_train_score=True)

print("📊 cross_validate (Multiple Metrics):")
print("-" * 50)
for metric in scoring:
    train_key = f'train_{metric}'
    test_key = f'test_{metric}'
    print(f"{metric.upper():12s} — Train: {cv_results[train_key].mean():.4f} ± {cv_results[train_key].std():.4f} | "
          f"Test: {cv_results[test_key].mean():.4f} ± {cv_results[test_key].std():.4f}")

print("\n💡 cross_validate gives you both train and test scores — essential for detecting overfitting!")

In [ ]:
# 📊 Visualizing cross-validation score distributions
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

X, y = make_classification(n_samples=500, n_features=15, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Ridge Classifier': Ridge(random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_scores = {}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot of CV scores
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    all_scores[name] = scores

bp = axes[0].boxplot(all_scores.values(), labels=all_scores.keys(), patch_artist=True)
colors = ['lightcoral', 'lightgreen', 'lightblue']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[0].set_title('📊 CV Score Distribution by Model', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0.5, 1.0)

# Bar plot with error bars
means = [np.mean(scores) for scores in all_scores.values()]
stds = [np.std(scores) for scores in all_scores.values()]
x_pos = np.arange(len(models))

bars = axes[1].bar(x_pos, means, yerr=stds, capsize=5, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(models.keys(), rotation=15)
axes[1].set_title('📊 Mean CV Score ± Std Dev', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.5, 1.0)
axes[1].grid(True, alpha=0.3)

# Add value labels on bars
for bar, mean, std in zip(bars, means, stds):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01,
                f'{mean:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("📏 Model Comparison (5-Fold Stratified CV):")
print("-" * 50)
for name, scores in all_scores.items():
    print(f"{name:20s}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

---

## 6. Applying to Titanic Dataset (Classification) 🚢

Let's apply our evaluation toolkit to the classic Titanic survival prediction problem.

In [ ]:
# 📦 Load and prepare Titanic dataset
try:
    titanic = sns.load_dataset('titanic')
    print("✅ Loaded Titanic dataset from seaborn")
except:
    # Fallback: create synthetic titanic-like data
    np.random.seed(42)
    n = 891
    titanic = pd.DataFrame({
        'survived': np.random.binomial(1, 0.38, n),
        'pclass': np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55]),
        'sex': np.random.choice(['male', 'female'], n, p=[0.65, 0.35]),
        'age': np.random.normal(29.7, 14.5, n),
        'fare': np.random.exponential(32.2, n),
        'embarked': np.random.choice(['S', 'C', 'Q'], n, p=[0.72, 0.19, 0.09]),
        'sibsp': np.random.poisson(0.5, n),
        'parch': np.random.poisson(0.38, n)
    })
    print("⚠️ Created synthetic Titanic-like dataset")

# Display basic info
print(f"\nDataset shape: {titanic.shape}")
print(f"\nSurvival rate: {titanic['survived'].mean():.3f}")
print(f"\nFirst few rows:")
print(titanic.head())

# Preprocessing
df = titanic.copy()
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df['embarked'] = df['embarked'].fillna('S')

# Encode categorical features
df['sex'] = LabelEncoder().fit_transform(df['sex'])
df['embarked'] = LabelEncoder().fit_transform(df['embarked'])

# Select features
features = ['pclass', 'sex', 'age', 'fare', 'sibsp', 'parch', 'embarked']
X_titanic = df[features]
y_titanic = df['survived']

print(f"\n✅ Features selected: {features}")
print(f"✅ Target: survived")
print(f"✅ Class distribution: {dict(y_titanic.value_counts().sort_index())}")

In [ ]:
# 🚢 Titanic: Full Evaluation Pipeline
from sklearn.pipeline import Pipeline

# Create pipeline with scaling
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Define cross-validation strategy (stratified for class imbalance)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Comprehensive evaluation
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_results = cross_validate(pipeline, X_titanic, y_titanic, cv=cv, scoring=scoring,
                            return_train_score=True)

# Display results in a clean table
results_df = pd.DataFrame({
    'Metric': [m.upper() for m in scoring],
    'Train Mean': [cv_results[f'train_{m}'].mean() for m in scoring],
    'Train Std': [cv_results[f'train_{m}'].std() for m in scoring],
    'Test Mean': [cv_results[f'test_{m}'].mean() for m in scoring],
    'Test Std': [cv_results[f'test_{m}'].std() for m in scoring]
})

print("📊 Titanic Survival Prediction — Cross-Validation Results")
print("=" * 65)
print(results_df.to_string(index=False, float_format='%.4f'))

# Check for overfitting
print("\n🔍 Overfitting Analysis:")
for metric in scoring:
    train_mean = cv_results[f'train_{metric}'].mean()
    test_mean = cv_results[f'test_{metric}'].mean()
    gap = train_mean - test_mean
    status = "⚠️ Possible overfitting" if gap > 0.1 else "✅ Good generalization"
    print(f"   {metric.upper():12s}: Train-Test gap = {gap:.4f} {status}")

In [ ]:
# 🚢 Titanic: Confusion Matrix and ROC Curve
# Train on full data for visualization
X_train, X_test, y_train, y_test = train_test_split(X_titanic, y_titanic, 
                                                    test_size=0.2, random_state=42, stratify=y_titanic)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Did Not Survive', 'Survived'],
            yticklabels=['Did Not Survive', 'Survived'],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title('📊 Titanic: Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.2, color='darkorange')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('📈 Titanic: ROC Curve', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Final Test Set Performance:")
print(f"   Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"   Precision: {precision_score(y_test, y_pred):.4f}")
print(f"   Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"   F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"   ROC-AUC:   {roc_auc:.4f}")

---

## 7. Applying to House Prices Dataset (Regression) 🏠

Now let's evaluate regression models using appropriate metrics.

In [ ]:
# 📦 Load and prepare California Housing dataset (built-in sklearn)
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
y_housing = housing.target

print("✅ California Housing Dataset Loaded")
print(f"   Shape: {X_housing.shape}")
print(f"   Features: {list(housing.feature_names)}")
print(f"   Target: Median House Value (in $100,000s)")
print(f"\nFirst 5 rows:")
print(X_housing.head())
print(f"\nTarget statistics:")
print(pd.Series(y_housing).describe())

In [ ]:
# 🏠 Regression Evaluation with Multiple Metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

# Define models
reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

# Regression scoring metrics
reg_scoring = ['neg_mean_absolute_error', 'neg_mean_squared_error', 'neg_root_mean_squared_error', 'r2']

# Evaluate each model
reg_results = {}
for name, model in reg_models.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', model)
    ])
    
    cv_results = cross_validate(pipeline, X_housing, y_housing, cv=5, 
                                scoring=reg_scoring, return_train_score=True)
    reg_results[name] = cv_results

# Create comparison table
comparison_data = []
for name, results in reg_results.items():
    row = {
        'Model': name,
        'MAE': -np.mean(results['test_neg_mean_absolute_error']),
        'MSE': -np.mean(results['test_neg_mean_squared_error']),
        'RMSE': -np.mean(results['test_neg_root_mean_squared_error']),
        'R²': np.mean(results['test_r2'])
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print("📊 Regression Model Comparison (5-Fold CV)")
print("=" * 60)
print(comparison_df.to_string(index=False, float_format='%.4f'))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² Comparison
models = comparison_df['Model']
r2_scores = comparison_df['R²']
colors = ['lightcoral', 'lightgreen', 'lightblue']

bars = axes[0].bar(models, r2_scores, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('📊 R² Score Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('R² Score')
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)
for bar, score in zip(bars, r2_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

# RMSE Comparison
rmse_scores = comparison_df['RMSE']
bars = axes[1].bar(models, rmse_scores, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_title('📊 RMSE Comparison (Lower is Better)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('RMSE ($100,000s)')
axes[1].grid(True, alpha=0.3)
for bar, score in zip(bars, rmse_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("   • R² = 1.0: Perfect prediction")
print("   • R² = 0.0: Model predicts the mean (no better than baseline)")
print("   • RMSE: Same units as target — directly interpretable")
print("   • MAE: Robust to outliers — good for skewed distributions")

---

## 8. Overfitting vs Underfitting Detection 🔍

Learning curves are one of the most powerful tools for diagnosing model behavior.

- **Overfitting**: Large gap between training and validation scores
- **Underfitting**: Both scores are low and close together
- **Good fit**: Both scores are high and close together

In [ ]:
# 📈 Learning Curves: Detecting Overfitting and Underfitting
from sklearn.model_selection import learning_curve

# Create datasets with different complexity
X_simple, y_simple = make_classification(n_samples=1000, n_features=2, n_informative=2,
                                         n_redundant=0, random_state=42)
X_complex, y_complex = make_classification(n_samples=1000, n_features=50, n_informative=5,
                                           n_redundant=45, random_state=42)

# Models with different complexity
models = {
    'Underfitting (High Bias)': LogisticRegression(max_iter=1000, random_state=42),
    'Good Fit': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Overfitting (High Variance)': RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, model) in enumerate(models.items()):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_complex, y_complex, cv=5, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy', random_state=42
    )
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    
    axes[idx].plot(train_sizes, train_mean, 'o-', color='blue', label='Training Score')
    axes[idx].fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    
    axes[idx].plot(train_sizes, val_mean, 'o-', color='green', label='Validation Score')
    axes[idx].fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='green')
    
    axes[idx].set_title(f'{name}', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Training Set Size')
    axes[idx].set_ylabel('Accuracy')
    axes[idx].legend(loc='best')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim(0.4, 1.05)

plt.suptitle('📈 Learning Curves: Diagnosing Model Behavior', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Learning Curve Interpretation:")
print("   🔴 Underfitting: Both curves low, close together → Increase model complexity")
print("   🟢 Good Fit: Both curves high, converging → Optimal model complexity")
print("   🟠 Overfitting: Training high, validation much lower → Reduce complexity or add data")

In [ ]:
# 📈 Validation Curves: Effect of Hyperparameters
from sklearn.model_selection import validation_curve

# Examine effect of max_depth on Random Forest
param_range = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30]
X_val, y_val = make_classification(n_samples=1000, n_features=20, random_state=42)

train_scores, val_scores = validation_curve(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_val, y_val, param_name='max_depth', param_range=param_range,
    cv=5, scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(param_range, train_mean, 'o-', color='blue', label='Training Score', linewidth=2)
plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')

plt.plot(param_range, val_mean, 'o-', color='green', label='Validation Score', linewidth=2)
plt.fill_between(param_range, val_mean - val_std, val_mean + val_std, alpha=0.1, color='green')

# Mark optimal region
optimal_idx = np.argmax(val_mean)
plt.axvline(param_range[optimal_idx], color='red', linestyle='--', alpha=0.7,
            label=f'Optimal max_depth = {param_range[optimal_idx]}')

plt.title('📈 Validation Curve: Random Forest max_depth', fontsize=14, fontweight='bold')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.ylim(0.5, 1.05)
plt.show()

print(f"📊 Validation Curve Results:")
print(f"   Optimal max_depth: {param_range[optimal_idx]} (Validation Accuracy: {val_mean[optimal_idx]:.4f})")
print(f"\n💡 Key Insight:")
print(f"   • Low max_depth: Underfitting (both scores low)")
print(f"   • High max_depth: Overfitting (training ↑, validation ↓)")
print(f"   • Sweet spot: Where validation score peaks before diverging")

---

## 9. Best Practices for Reliable Evaluation ✅

### The Golden Rules of Model Evaluation:

1. **Never peek at the test set** — data leakage destroys validity
2. **Use stratification for imbalanced data** — ensures representative folds
3. **Report multiple metrics** — no single metric tells the whole story
4. **Check variance across folds** — high variance indicates unstable models
5. **Always compare against a baseline** — random classifier or mean prediction
6. **Use confidence intervals** — point estimates can be misleading
7. **Match CV strategy to data** — time series needs temporal splits
8. **Scale features within CV** — prevent data leakage from preprocessing
9. **Document your evaluation protocol** — reproducibility is key
10. **Consider the business cost** — optimize for the metric that matters most

In [ ]:
# ✅ Best Practices: Proper Pipeline with Cross-Validation
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("✅ BEST PRACTICE: Pipeline + Cross-Validation")
print("=" * 50)

# ❌ WRONG: Scaling before splitting (data leakage!)
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)  # DON'T DO THIS!
# scores = cross_val_score(model, X_scaled, y, cv=5)

# ✅ CORRECT: Scale inside the pipeline
proper_pipeline = Pipeline([
    ('scaler', StandardScaler()),      # Fits on train, transforms both train and test
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

X_demo, y_demo = make_classification(n_samples=500, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(proper_pipeline, X_demo, y_demo, cv=cv, scoring='accuracy')

print(f"✅ Correct approach — Pipeline CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
print(f"\n💡 Why this matters:")
print(f"   • Pipeline ensures scaler only sees training data in each fold")
print(f"   • Prevents information leakage from test to train")
print(f"   • Guarantees fair, unbiased performance estimates")
print(f"   • Essential for any preprocessing step (scaling, imputation, encoding)")

---

## 10. Nested Cross-Validation (Advanced Intro) 🎯

Nested Cross-Validation is the gold standard for unbiased model selection AND performance estimation.

**Problem**: Using the same CV for both hyperparameter tuning and performance estimation leads to optimistic bias.

**Solution**: 
- **Outer loop**: Estimates generalization performance
- **Inner loop**: Selects best hyperparameters

This gives you an unbiased estimate of how your model (with hyperparameter tuning) will perform on truly unseen data.

In [ ]:
# 🎯 Nested Cross-Validation Demonstration
from sklearn.model_selection import GridSearchCV

print("🎯 Nested Cross-Validation")
print("=" * 50)

# Create dataset
X_nested, y_nested = make_classification(n_samples=500, n_features=20, random_state=42)

# Define parameter grid
param_grid = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2']
}

# Inner CV: Hyperparameter tuning
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Outer CV: Performance estimation
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(solver='liblinear', max_iter=1000, random_state=42))
])

# Nested CV
nested_scores = cross_val_score(
    GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring='accuracy', n_jobs=-1),
    X_nested, y_nested, cv=outer_cv, scoring='accuracy'
)

print(f"📊 Nested CV Results:")
print(f"   Scores per outer fold: {nested_scores}")
print(f"   Mean Accuracy: {nested_scores.mean():.4f} ± {nested_scores.std():.4f}")
print(f"\n💡 This is the UNBIASED estimate of your model's true performance!")
print(f"   It accounts for both model selection variance and generalization variance.")
print(f"\n⚠️ Note: Nested CV is computationally expensive but essential for:")
print(f"   • Research papers and benchmarks")
print(f"   • High-stakes applications (medical, financial)")
print(f"   • Small datasets where every sample matters")

---

## 🛠️ Hands-On Exercises

Put your new skills to the test! Complete these exercises to solidify your understanding of model evaluation and cross-validation.

### Exercise 1: Basic K-Fold Cross-Validation
Implement 10-fold cross-validation on the Iris dataset and compare the results with a single train-test split.

### Exercise 2: Stratified K-Fold on Imbalanced Data
Create a highly imbalanced synthetic dataset (90% vs 10%) and compare StratifiedKFold with regular KFold using F1-score.

### Exercise 3: Compute All Classification Metrics
Train a classifier on any dataset and manually compute Accuracy, Precision, Recall, F1-Score, and ROC-AUC. Verify your calculations match sklearn's built-in functions.

### Exercise 4: Plot Learning Curves
Use `learning_curve` to plot training and validation curves for three different models on the same dataset. Identify which model is overfitting, underfitting, or well-fitted.

### Exercise 5: Compare CV vs Train-Test Split
Run 20 different random train-test splits and compare the variance of scores against 5-fold cross-validation on the same model and dataset.

### Exercise 6: Regression Metrics Pipeline
Build a complete evaluation pipeline for a regression problem that reports MAE, MSE, RMSE, and R² using cross-validation. Include a visualization comparing models.

### Exercise 7: Confusion Matrix Interpretation
Generate a confusion matrix for a binary classification problem and calculate Sensitivity, Specificity, PPV, and NPV manually from the matrix values.

### Exercise 8: ROC Curve Comparison
Train three different classifiers on the same dataset and plot their ROC curves on the same graph. Calculate and compare their AUC scores.

### Exercise 9: Nested CV for Hyperparameter Tuning
Implement nested cross-validation to find the optimal `max_depth` for a Random Forest classifier and report the unbiased performance estimate.

### Exercise 10: Complete Evaluation Report
Choose any dataset (classification or regression) and create a comprehensive evaluation report including:
- Data exploration and preprocessing
- Multiple model comparison with CV
- Learning curves
- Best model selection with justification
- Final test set evaluation with all relevant metrics

---

## Solutions (Review After Attempting) 🔑

Below are detailed solutions with explanations. Try the exercises first before reviewing!

### Solution 1: Basic K-Fold Cross-Validation

```python
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
import numpy as np

# Load data
iris = load_iris()
X, y = iris.data, iris.target

# Single train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
split_score = model.score(X_test, y_test)

# 10-fold CV
cv_scores = cross_val_score(model, X, y, cv=10)

print(f"Single split accuracy: {split_score:.4f}")
print(f"10-fold CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print("\n💡 CV gives a more robust estimate by averaging over multiple splits.")
```

### Solution 2: Stratified K-Fold on Imbalanced Data

```python
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Highly imbalanced dataset
X, y = make_classification(n_samples=1000, weights=[0.9, 0.1], random_state=42)
model = LogisticRegression(max_iter=1000, random_state=42)

# Compare strategies
kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

kf_scores = cross_val_score(model, X, y, cv=kf, scoring='f1')
skf_scores = cross_val_score(model, X, y, cv=skf, scoring='f1')

print(f"K-Fold F1:      {kf_scores.mean():.4f} ± {kf_scores.std():.4f}")
print(f"Stratified F1:  {skf_scores.mean():.4f} ± {skf_scores.std():.4f}")
print("\n💡 StratifiedKFold preserves class proportions, giving more reliable estimates on imbalanced data.")
```

### Solution 3: Compute All Classification Metrics

```python
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

# Assume y_true and y_pred are available
# Manual calculation from confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Manual metrics
manual_accuracy = (tp + tn) / (tp + tn + fp + fn)
manual_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
manual_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
manual_f1 = 2 * (manual_precision * manual_recall) / (manual_precision + manual_recall)

# Verify with sklearn
print(f"Accuracy:  Manual={manual_accuracy:.4f}, Sklearn={accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: Manual={manual_precision:.4f}, Sklearn={precision_score(y_test, y_pred):.4f}")
print(f"Recall:    Manual={manual_recall:.4f}, Sklearn={recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  Manual={manual_f1:.4f}, Sklearn={f1_score(y_test, y_pred):.4f}")
```

### Solution 4: Plot Learning Curves

```python
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

models = {
    'Underfitting': LogisticRegression(max_iter=1000),
    'Good Fit': RandomForestClassifier(n_estimators=50, max_depth=5),
    'Overfitting': RandomForestClassifier(n_estimators=50, max_depth=None)
}

for name, model in models.items():
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
    )
    plt.plot(train_sizes, np.mean(train_scores, axis=1), label=f'{name} (train)')
    plt.plot(train_sizes, np.mean(val_scores, axis=1), label=f'{name} (val)')

plt.xlabel('Training Size')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Learning Curves')
plt.show()
```

### Solution 5: Compare CV vs Train-Test Split

```python
split_scores = []
for seed in range(20):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    model.fit(X_tr, y_tr)
    split_scores.append(model.score(X_te, y_te))

cv_scores = cross_val_score(model, X, y, cv=5)

print(f"Train-Test Split: {np.mean(split_scores):.4f} ± {np.std(split_scores):.4f}")
print(f"5-Fold CV:        {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"\n💡 Train-test split variance is typically much higher than CV variance.")
```

### Solution 6: Regression Metrics Pipeline

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate

models = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(),
    'RF': RandomForestRegressor(n_estimators=100)
}

scoring = ['neg_mean_absolute_error', 'neg_mean_squared_error', 'r2']

for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    results = cross_validate(pipe, X, y, cv=5, scoring=scoring)
    print(f"{name}: MAE={-results['test_neg_mean_absolute_error'].mean():.4f}, "
          f"R²={results['test_r2'].mean():.4f}")
```

### Solution 7: Confusion Matrix Interpretation

```python
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)      # Same as Recall / TPR
specificity = tn / (tn + fp)      # True Negative Rate
ppv = tp / (tp + fp)              # Positive Predictive Value (Precision)
npv = tn / (tn + fn)              # Negative Predictive Value

print(f"Sensitivity (TPR): {sensitivity:.4f}")
print(f"Specificity (TNR): {specificity:.4f}")
print(f"PPV (Precision):   {ppv:.4f}")
print(f"NPV:               {npv:.4f}")
```

### Solution 8: ROC Curve Comparison

```python
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'SVM': SVC(probability=True)
}

plt.figure(figsize=(8, 6))
for name, model in models.items():
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()
```

### Solution 9: Nested CV for Hyperparameter Tuning

```python
from sklearn.model_selection import GridSearchCV, cross_val_score

param_grid = {'max_depth': [3, 5, 7, 10, 15, 20, None]}
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

nested_scores = cross_val_score(
    GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=inner_cv),
    X, y, cv=outer_cv, scoring='accuracy'
)

print(f"Nested CV Accuracy: {nested_scores.mean():.4f} ± {nested_scores.std():.4f}")
print("💡 This is the unbiased estimate of tuned model performance.")
```

### Solution 10: Complete Evaluation Report

```python
# Framework for a complete evaluation report
"""
1. Data Exploration
   - Load and inspect data
   - Check for missing values, outliers, class imbalance
   - Visualize distributions and correlations

2. Preprocessing
   - Handle missing values
   - Encode categorical variables
   - Scale features (inside pipeline!)

3. Model Comparison
   - Define candidate models
   - Use cross_validate with multiple metrics
   - Create comparison tables and visualizations

4. Learning Curves
   - Plot for top 2-3 models
   - Identify overfitting/underfitting

5. Hyperparameter Tuning
   - GridSearchCV or RandomizedSearchCV
   - Use proper validation strategy

6. Final Evaluation
   - Train best model on full training data
   - Evaluate on held-out test set
   - Report all relevant metrics with confidence intervals
   - Include confusion matrix and ROC curve for classification
"""
```